In [1]:
import kagglehub, os, glob, hashlib

IMG  = kagglehub.dataset_download("ashwanisingh997/ham10000-images-1229692")
SPL  = kagglehub.dataset_download("ashwanisingh997/ham10000-splits-1229692")
CODE = kagglehub.dataset_download("ashwanisingh997/thesis-code-1229692")
SYN  = kagglehub.dataset_download("ashwanisingh997/run1-synthetic-headline-1229692")
for k, v in [("IMG", IMG), ("SPL", SPL), ("CODE", CODE), ("SYN", SYN)]:
    print(f"{k}: {v}")

METRICS_SHA = "602747135A78568EAF96E64DB70A7C43CAC6F8527089DBE26711B0D834F05E17"
h = hashlib.sha256(open(f"{CODE}/metrics.py", "rb").read()).hexdigest().upper()
assert h == METRICS_SHA, f"metrics.py is not v4 (f568fe0): {h}"
print("code: metrics.py v4 (f568fe0) confirmed")

SYN_ROOT = f"{SYN}/synthetic/run1_cloud"
for arm in ["gan_final", "lora_final"]:
    for c in ["akiec", "bcc", "df", "mel", "vasc"]:
        n = len(glob.glob(f"{SYN_ROOT}/{arm}/{c}/*.png"))
        assert n == 1000, (arm, c, n)
print("synthetic set: 2 arms x 5 classes x 1000 confirmed")

IMG: /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
SPL: /kaggle/input/datasets/ashwanisingh997/ham10000-splits-1229692
CODE: /kaggle/input/datasets/ashwanisingh997/thesis-code-1229692
SYN: /kaggle/input/datasets/ashwanisingh997/run1-synthetic-headline-1229692
code: metrics.py v4 (f568fe0) confirmed
synthetic set: 2 arms x 5 classes x 1000 confirmed


In [2]:
import sys, shutil, torch, pandas as pd

WORK = "/kaggle/working/repo"
os.makedirs(f"{WORK}/src", exist_ok=True)
os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    dst = f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}"
    shutil.copy(f"{CODE}/{f}", dst)
sys.path.insert(0, WORK)
os.chdir(WORK)

print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
assert torch.__version__.startswith("2.11"), "environment differs from the generator and sampling sessions"

from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash="f568fe0")

from src.data import HAM10000, make_loader, build_image_index, assert_no_synthetic_leakage, AUG_CLASSES
assert_no_synthetic_leakage(f"{SPL}/val.csv", f"{SPL}/test.csv")
index = build_image_index(IMG)
NW = 4

val_ds  = HAM10000(f"{SPL}/val.csv",  IMG, size=128, augment=False, index=index)
test_ds = HAM10000(f"{SPL}/test.csv", IMG, size=128, augment=False, index=index)
val_loader  = make_loader(val_ds,  batch_size=32, shuffle=False, num_workers=NW)
test_loader = make_loader(test_ds, batch_size=32, shuffle=False, num_workers=NW)
print(f"val {len(val_ds)}  test {len(test_ds)}")

2.11.0+cu128 | GPU: Tesla T4
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.11.0+cu128 | cuda 12.8
[registry] writing to /kaggle/working/results/run1_cloud
[data] no synthetic leakage in 2 partition(s)
[data] indexed 10015 images under /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
val 1532  test 1502


In [3]:
def synthetic_rows(arm, n_per_class=1000):
    rows = []
    for c in AUG_CLASSES:
        files = sorted(glob.glob(f"{SYN_ROOT}/{arm}/{c}/*.png"))[:n_per_class]
        assert len(files) == n_per_class, (arm, c, len(files))
        rows += [{"image_id": os.path.basename(f)[:-4], "dx": c, "path": f, "synthetic": True}
                 for f in files]
    return pd.DataFrame(rows)

CONDS   = ["A_baseline_rerun", "B_classical_rerun", "C_gan", "D_diffusion"]
EXTRA   = {"A_baseline_rerun": None, "B_classical_rerun": None,
           "C_gan": synthetic_rows("gan_final"), "D_diffusion": synthetic_rows("lora_final")}
AUGMENT = {"A_baseline_rerun": False, "B_classical_rerun": True, "C_gan": False, "D_diffusion": False}
for c in CONDS:
    n_syn = 0 if EXTRA[c] is None else len(EXTRA[c])
    print(f"{c:18s} augment={AUGMENT[c]!s:5s} synthetic rows={n_syn}")

A_baseline_rerun   augment=False synthetic rows=0
B_classical_rerun  augment=True  synthetic rows=0
C_gan              augment=False synthetic rows=5000
D_diffusion        augment=False synthetic rows=5000


In [4]:
from src.utils import set_seed
from src.classifier import build_model, train_classifier
from src.metrics import evaluate, save_metrics

SMOKE  = False   # True: 1 epoch, seed 42 only, as a test. False: the real run.
SEEDS  = [42] if SMOKE else [42, 43, 44]
EPOCHS = 1 if SMOKE else 30
PRE    = "SMOKE_" if SMOKE else ""

for cond in CONDS:
    for seed in SEEDS:
        name = f"{PRE}{cond}_seed{seed}"
        if os.path.exists(f"{root}/metrics/{name}.json"):
            print(f"skip {name}: already done"); continue
        print(f"\n{'='*60}\nRUN 1 - {cond} - SEED {seed}\n{'='*60}")
        set_seed(seed)
        tr = HAM10000(f"{SPL}/train.csv", IMG, size=128, augment=AUGMENT[cond],
                      index=index, extra_rows=EXTRA[cond])
        expect = 6981 + (0 if EXTRA[cond] is None else 5000)
        assert len(tr) == expect, (cond, len(tr), expect)
        print(f"train images: {len(tr)}")
        tr_ld = make_loader(tr, batch_size=32, shuffle=True, num_workers=NW)
        m, hist = train_classifier(build_model(), tr_ld, val_loader,
                                   epochs=EPOCHS, lr=1e-4, weight_decay=1e-4, patience=7,
                                   tag=name, log_dir=f"{root}/logs")
        r = evaluate(m, test_loader, bootstrap=True, seed=seed)
        save_metrics(r, name, root=str(root))
        del m; torch.cuda.empty_cache()
print("ALL RUNS COMPLETE")


RUN 1 - A_baseline_rerun - SEED 42
train images: 6981
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]


ep 00  loss 0.7970  val macroF1 0.4811  val acc 0.7761
ep 01  loss 0.3912  val macroF1 0.5379  val acc 0.8029
ep 02  loss 0.1832  val macroF1 0.5871  val acc 0.7898
ep 03  loss 0.1065  val macroF1 0.5980  val acc 0.7774
ep 04  loss 0.0713  val macroF1 0.5677  val acc 0.7957
ep 05  loss 0.0434  val macroF1 0.6125  val acc 0.8055
ep 06  loss 0.0284  val macroF1 0.6318  val acc 0.8107
ep 07  loss 0.0310  val macroF1 0.5801  val acc 0.7839
ep 08  loss 0.0338  val macroF1 0.6245  val acc 0.8107
ep 09  loss 0.0181  val macroF1 0.5890  val acc 0.8055
ep 10  loss 0.0159  val macroF1 0.6126  val acc 0.8022
ep 11  loss 0.0173  val macroF1 0.6176  val acc 0.8127
ep 12  loss 0.0112  val macroF1 0.6158  val acc 0.8048
ep 13  loss 0.0103  val macroF1 0.5769  val acc 0.7768
Early stopping at epoch 13 (best macro-F1 0.6318)
              precision    recall  f1-score   support

       akiec      0.485     0.635     0.550        52
         bcc      0.562     0.577     0.569        71
         bkl     

In [5]:
import json, numpy as np

METRICS = ["macro_f1", "melanoma_recall", "balanced_accuracy", "accuracy",
           "macro_auroc", "macro_precision", "macro_auprc"]
rows = []
for cond in CONDS:
    rs = [json.load(open(f"{root}/metrics/{PRE}{cond}_seed{s}.json")) for s in SEEDS]
    for m in METRICS:
        v = [r[m] for r in rs]
        rows.append({"condition": cond, "metric": m, "mean": np.mean(v),
                     "sd": np.std(v, ddof=1) if len(v) > 1 else float("nan"),
                     "values": [round(x, 4) for x in v]})
summ = pd.DataFrame(rows).round(4)
summ.to_csv(f"{root}/metrics/{PRE}ABCD_summary.csv", index=False)
print(summ.pivot(index="metric", columns="condition", values="mean").to_string())

npz = sorted(glob.glob(f"{root}/metrics/{PRE}*_predictions.npz"))
assert len(npz) == len(CONDS) * len(SEEDS), len(npz)
for f in npz:
    z = np.load(f)
    assert z["probs"].shape == (1502, 7) and len(z["y_true"]) == 1502, f
print(f"{len(npz)} prediction files, each 1502 test images x 7 classes")

shutil.make_archive("/kaggle/working/run1_ABCD_results", "zip", "/kaggle/working/results")
print("saved: /kaggle/working/run1_ABCD_results.zip")

condition          A_baseline_rerun  B_classical_rerun   C_gan  D_diffusion
metric                                                                     
accuracy                     0.7960             0.8058  0.8009       0.7976
balanced_accuracy            0.5979             0.6609  0.6053       0.6099
macro_auprc                  0.6744             0.7075  0.6767       0.6936
macro_auroc                  0.9349             0.9480  0.9370       0.9388
macro_f1                     0.6199             0.6625  0.6329       0.6362
macro_precision              0.6612             0.6778  0.6790       0.6863
melanoma_recall              0.4691             0.4451  0.4930       0.4092
12 prediction files, each 1502 test images x 7 classes
saved: /kaggle/working/run1_ABCD_results.zip
